In [93]:
from sklearn.datasets import make_classification
import torch

# make_classification - generates FAKE data for testing/learning
X, y = make_classification(
    n_samples=10,       # 10 rows 
    n_features=2,       # 2 columns (features)
    n_informative=2,    # both features carry real signal (not noise)
    n_redundant=0,      # no useless duplicate features
    n_classes=2,        # binary classification (0 or 1)
    random_state=42    
)

In [94]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [95]:
X.shape

(10, 2)

In [96]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [97]:
y.shape

(10,)

In [98]:
X.dtype

dtype('float64')

In [99]:
y.dtype

dtype('int64')

In [100]:
# convert numpy -> tensors with the correct dtypes
X = torch.tensor(X, dtype=torch.float32)   # float32 - required by nn.Linear
y = torch.tensor(y, dtype=torch.long)      # long (int) - required by CrossEntropyLoss

In [101]:
X.dtype

torch.float32

In [102]:
y.dtype

torch.int64

In [103]:
from torch.utils.data import Dataset, DataLoader


class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
      # tells PyTorch HOW MANY samples exist
      # DataLoader needs this to know when an epoch ends
      return self.features.shape[0]   

  def __getitem__(self, index):
      # tells PyTorch HOW TO FETCH one sample by its position
      # DataLoader calls this repeatedly and stacks the results into a batch
      return self.features[index], self.labels[index]   # returns a (features, label) tuple

In [104]:
dataset=CustomDataset(X,y)

In [105]:
len(dataset)

10

In [106]:
dataset[6]

(tensor([ 1.7273, -1.1858]), tensor(1))

In [107]:
# DataLoader - takes the Dataset and serves it in BATCHES automatically
dataloader = DataLoader(
    dataset,
    batch_size=2,    # 2 samples per batch (10 samples / 2 = 5 batches)
    shuffle=True     # randomize the order each epoch 
)

In [108]:
# loop through the batches - DataLoader stacks individual samples into batch tensors
for batch_features, batch_labels in dataloader:
    print(batch_features)  
    print(batch_labels)     
    print("-"*50)
    
# 10 samples / batch_size 2 = 5 batches
# NOTE: we only wrote __getitem__ to return ONE sample -
# DataLoader calls it repeatedly and STACKS the results into a batch for us

tensor([[ 1.8997,  0.8344],
        [-1.9629, -0.9923]])
tensor([1, 0])
--------------------------------------------------
tensor([[ 1.0683, -0.9701],
        [ 1.7774,  1.5116]])
tensor([1, 1])
--------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [-0.9382, -0.5430]])
tensor([1, 1])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-1.1402, -0.8388]])
tensor([0, 0])
--------------------------------------------------
tensor([[-0.5872, -1.9717],
        [-0.7206, -0.9606]])
tensor([0, 0])
--------------------------------------------------


## Why Use Batches?

Feeding data in batches instead of all at once gives us three advantages:

**1. Memory efficiency**  
You can't fit a million samples into RAM or GPU memory at once. Batches let you train on huge datasets with limited memory.

**2. Faster convergence**   
The weights update **once per batch**, not once per epoch. With 455 samples and `batch_size=32`, that's ~14 weight updates per epoch instead of 1.

**3. The sweet spot between two extremes** 

| Approach | Batch size | Problem |
|----------|-----------|---------|
| Gradient Descent | ALL data | memory-hungry |
| SGD | 1 sample | very noisy, slow |
| **Mini-batch**  | 32, 64, 128... | **balanced — best of both** |

> `DataLoader(batch_size=32)` is exactly **Mini-batch SGD** — the approach used in practice.

## Additional DataLoader Options

Two parameters worth knowing beyond `batch_size` and `shuffle`:

**`drop_last`** — handles the uneven final batch  
If your data doesn't divide evenly, the last batch is smaller:
```
100 samples, batch_size=32 → batches of [32, 32, 32, 4]   ← last one has only 4!
```
```python
DataLoader(dataset, batch_size=32, drop_last=True)   # discards the small last batch
```
A tiny last batch can cause problems (especially with BatchNorm, which needs several samples to compute batch statistics).

**`num_workers`** — parallel loading for speed
```python
DataLoader(dataset, batch_size=32, num_workers=2)   # load batches using 2 CPU processes
```
Loads batches in parallel → faster data loading. Matters for images and large datasets; not needed for small tabular data. Default is `0` (single process).

# A Note About DataLoader Parallelization (`num_workers=4`)

## Example Setup

- Total samples = **10,000**
- Batch size = **32**
- Number of workers (`num_workers`) = **4**
- Total batches per epoch ≈ **312** (`10000 / 32 ≈ 312`)

---

# Step 1: Sampler Creates All Batches (Main Process)

Before training starts:

1. All **10,000 sample indices** are created.
2. If `shuffle=True`, these indices are shuffled randomly.
3. The indices are divided into **312 batches**, each containing **32 indices**.
4. These batches are placed in a queue, ready for the workers.

**Think of it like this:**

```
10000 indices
        │
   Shuffle (optional)
        │
Split into 312 batches
        │
Ready for workers
```

---

# Step 2: Workers Start Loading Data

When the training loop starts:

```python
for batch_data, batch_labels in dataloader:
    # Training code
```

The DataLoader immediately gives the first four batches to the four workers.

```
Worker 1 → Batch 1
Worker 2 → Batch 2
Worker 3 → Batch 3
Worker 4 → Batch 4
```

Each worker:

- Reads the assigned sample indices.
- Calls `__getitem__()` for every index.
- Applies any transformations.
- Combines all samples into one batch using `collate_fn`.

---

# Step 3: First Batch is Returned

Whichever worker finishes first sends its batch back to the **main process**.

Example:

```
Worker 1 finishes first
        │
Returns Batch 1
        │
Training loop receives it
```

Now your code gets:

```python
batch_data, batch_labels
```

---

# Step 4: Model Starts Training

The main process now performs:

1. Forward pass
2. Loss calculation
3. Backward pass
4. Optimizer step

Example:

```python
outputs = model(batch_data)
loss = criterion(outputs, batch_labels)

loss.backward()
optimizer.step()
```

### Meanwhile...

The other workers **do not wait**.

While the model is training on Batch 1:

- Worker 2 is preparing Batch 2
- Worker 3 is preparing Batch 3
- Worker 4 is preparing Batch 4

This saves a lot of waiting time.

---

# Step 5: Continuous Loading

Whenever a worker finishes its current batch, it immediately starts the next one.

Example:

```
Worker 1 → Batch 1 ✓
          ↓
        Batch 5

Worker 2 → Batch 2 ✓
          ↓
        Batch 6

Worker 3 → Batch 3 ✓
          ↓
        Batch 7

Worker 4 → Batch 4 ✓
          ↓
        Batch 8
```

So there are always up to **4 batches being prepared at the same time**.

---

# Step 6: Training Loop Continues

Your training loop stays very simple:

```python
for batch_data, batch_labels in dataloader:

    # Forward pass
    outputs = model(batch_data)

    # Compute loss
    loss = criterion(outputs, batch_labels)

    # Backward pass
    loss.backward()

    # Update weights
    optimizer.step()
```

You don't need to manage the workers yourself.

The DataLoader automatically keeps preparing the next batches in the background.

---

# Step 7: End of the Epoch

After all **312 batches** have been processed:

- Every sample has been used once.
- The DataLoader has no more batches to return.
- The epoch ends.

If `shuffle=True`:

1. The indices are shuffled again.
2. New batches are created.
3. The same process repeats for the next epoch.

---

# Complete Workflow

```
Dataset (10,000 samples)
        │
        ▼
Sampler creates indices
        │
        ▼
Shuffle (if enabled)
        │
        ▼
Create 312 batches
        │
        ▼
Assign first 4 batches
        │
 ┌──────┼──────┬──────┐
 ▼      ▼      ▼      ▼
W1     W2     W3     W4
 │      │      │      │
Load   Load   Load   Load
 │      │      │      │
 └──────┼──────┴──────┘
        ▼
First completed batch
        ▼
Main Process
        ▼
Forward Pass
        ▼
Loss
        ▼
Backward Pass
        ▼
Optimizer Step
        ▼
Next Batch (already prepared)
        ▼
Repeat until all batches are processed
```

---

# Key Points to Remember

- **Main Process**
  - Creates and shuffles indices.
  - Sends batches to workers.
  - Trains the model.

- **Workers**
  - Load data.
  - Call `__getitem__()`.
  - Apply transforms.
  - Create batches.
  - Return ready-to-use batches.

- **Benefit of `num_workers > 0`**
  - Data loading happens **in parallel** with model training.
  - The GPU/CPU spends less time waiting for data.
  - Training becomes faster, especially for large datasets.

# DataLoader Important Parameters

The `DataLoader` in PyTorch has several parameters that control **how data is loaded, batched, shuffled, and sent to the model**.

---

# 1. `dataset` (Required)

The dataset from which the DataLoader reads data.

It must be a class that inherits from `torch.utils.data.Dataset` and implements:

- `__len__()` → Returns the total number of samples.
- `__getitem__()` → Returns one sample for a given index.

### Example

```python
train_loader = DataLoader(dataset=train_dataset)
```

---

# 2. `batch_size`

Specifies **how many samples are loaded in one batch**.

### Example

```python
DataLoader(dataset, batch_size=32)
```

If the dataset has **100 samples**:

```
Batch 1 → 32 samples
Batch 2 → 32 samples
Batch 3 → 32 samples
Batch 4 → 4 samples
```

### Default

```python
batch_size = 1
```

### Advantages of Large Batch Size

- Faster GPU utilization.
- Fewer iterations per epoch.

### Disadvantages

- Uses more RAM/VRAM.
- May not fit into GPU memory.

---

# 3. `shuffle`

Controls whether the dataset is shuffled before every epoch.

### Example

```python
DataLoader(dataset, shuffle=True)
```

Without shuffle:

```
Epoch 1
1 2 3 4 5 6 ...

Epoch 2
1 2 3 4 5 6 ...
```

With shuffle:

```
Epoch 1
7 2 9 1 5 ...

Epoch 2
3 8 6 2 1 ...
```

### Why use `shuffle=True`?

- Prevents the model from learning the order of the data.
- Improves generalization.
- Commonly used for the training dataset.

---

# 4. `num_workers`

Specifies **how many worker processes load data in parallel**.

### Example

```python
DataLoader(dataset, num_workers=4)
```

If:

```
num_workers = 0
```

```
Main Process
     │
Load Data
     │
Train Model
```

Everything happens one after another.

If:

```
num_workers = 4
```

```
Worker 1 → Load Batch 1
Worker 2 → Load Batch 2
Worker 3 → Load Batch 3
Worker 4 → Load Batch 4

           ↓

Main Process trains the model
```

Data loading happens **in parallel**, making training faster.

### Rule of Thumb

- `0` → No parallel loading.
- `2–8` → Common values.
- Too many workers may increase CPU usage.

---

# 5. `pin_memory`

Used mainly when training on a **GPU (CUDA)**.

### Example

```python
DataLoader(dataset, pin_memory=True)
```

When `pin_memory=True`:

```
Dataset
    │
Pinned Memory (CPU)
    │
GPU
```

Data is copied from CPU to GPU faster.

### Use It When

- Training on NVIDIA GPU using CUDA.

### No Benefit

- CPU-only training.

---

# 6. `drop_last`

Determines what to do with the **last incomplete batch**.

### Example

Suppose:

```
Dataset = 32 samples
Batch Size = 10
```

Without `drop_last`:

```
Batch 1 → 10 ✓
Batch 2 → 10 ✓
Batch 3 → 10 ✓
Batch 4 → 2  ✓
```

All samples are used.

With:

```python
drop_last=True
```

```
Batch 1 → 10 ✓
Batch 2 → 10 ✓
Batch 3 → 10 ✓
Batch 4 → 2 ✗ (Dropped)
```

The last incomplete batch is ignored.

### Why use `drop_last=True`?

Some models (e.g., Batch Normalization) work better when **every batch has the same size**.

---

# 7. `collate_fn`

Controls **how individual samples are combined into a batch**.

By default:

```
Sample 1
Sample 2
Sample 3
Sample 4
      │
      ▼
One Batch
```

Default behavior:

```python
collate_fn = default_collate
```

### Custom `collate_fn`

Useful when:

- Images have different sizes.
- Sentences have different lengths.
- Custom batching is needed.

---

# 8. `sampler`

Controls **which samples are selected and in what order**.

Instead of using `shuffle=True`, you can define your own sampling strategy.

Example:

```
Dataset

1
2
3
4
5
6

Sampler decides:

4
2
6
1
5
3
```

Useful for:

- Imbalanced datasets.
- Random sampling.
- Weighted sampling.

---

# 9. `batch_sampler`

Controls **how batches are created**.

Normally:

```
batch_size = 32
```

PyTorch automatically creates batches.

With `batch_sampler`, you manually decide:

```
Batch 1 → Indices [5, 10, 22]

Batch 2 → Indices [3, 8, 17]

Batch 3 → Indices [40, 41, 42]
```

Usually used in advanced applications.

---

# Summary Table

| Parameter | Purpose | Common Value |
|-----------|---------|--------------|
| `dataset` | Source of data | Required |
| `batch_size` | Number of samples per batch | `32`, `64`, `128` |
| `shuffle` | Shuffle data every epoch | `True` (training) |
| `num_workers` | Number of parallel workers | `2–8` |
| `pin_memory` | Faster CPU → GPU transfer | `True` (GPU only) |
| `drop_last` | Drop incomplete last batch | `False` (default) |
| `collate_fn` | Combine samples into batches | Default or custom |
| `sampler` | Controls sample selection order | Optional |
| `batch_sampler` | Controls batch formation | Advanced |

---

# Quick Notes

- **`dataset`** → Provides the data.
- **`batch_size`** → Number of samples in one batch.
- **`shuffle`** → Randomizes data every epoch.
- **`num_workers`** → Loads data in parallel using multiple CPU processes.
- **`pin_memory`** → Speeds up CPU-to-GPU data transfer.
- **`drop_last`** → Removes the last incomplete batch.
- **`collate_fn`** → Combines individual samples into one batch.
- **`sampler`** → Decides which samples to pick.
- **`batch_sampler`** → Decides how batches are formed.